# **CSC2042S: Assignment 2**

## **Multinomial Logistic Regression for News Topic Classification**

**Name:** Sanele Hopewell Nkosi  
**Student Number:** nkssan033

In [20]:
%matplotlib inline
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets
import numpy as np
import random
import string

## **Question1:DATA PROCESSING**
The MasakhaNEWS training, development, and test datasets are loaded for English, isiXhosa, and chiShona. The headline and article text are combined, cleaned, and tokenised. The class labels are also converted into numerical values for use by the classification model.

In [10]:
import csv
def read_Masakha_News(file_path):
    with open(file_path, encoding="utf-8") as file:
        data = list(csv.DictReader(file, delimiter="\t"))
    
    return data

In [11]:
# English datasets
eng_train = read_Masakha_News("eng/train.tsv")
eng_dev = read_Masakha_News("eng/dev.tsv")
eng_test = read_Masakha_News("eng/test.tsv")

# isiXhosa datasets
xho_train = read_Masakha_News("xho/train.tsv")
xho_dev = read_Masakha_News("xho/dev.tsv")
xho_test = read_Masakha_News("xho/test.tsv")

# chiShona datasets
sna_train = read_Masakha_News("sna/train.tsv")
sna_dev = read_Masakha_News("sna/dev.tsv")
sna_test = read_Masakha_News("sna/test.tsv")

In [12]:
print("English")
print("Training:",len(eng_train))
print("Development:",len(eng_dev))
print("Test:",len(eng_test))

print("\nisiXhosa")
print("Training:",len(xho_train))
print("Development:",len(xho_dev))
print("Test:",len(xho_test))

print("\nchiShona")
print("Training:",len(sna_train))
print("Development:",len(sna_dev))
print("Test:",len(sna_test))

English
Training: 3309
Development: 472
Test: 948

isiXhosa
Training: 1032
Development: 147
Test: 297

chiShona
Training: 1288
Development: 185
Test: 369


In [13]:
print(eng_train[0].keys())

dict_keys(['category', 'headline', 'text', 'url'])


In [14]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())
    
    return text

In [17]:
def prepare_data(data):
    texts = []
    categories = []

    for article in data:
        combined_text = article["headline"] + " " + article["text"]
        combined_text = clean_text(combined_text)

        texts.append(combined_text)
        categories.append(article["category"])

    return texts, categories

In [21]:
# English
eng_train_texts, eng_train_categories = prepare_data(eng_train)
eng_dev_texts, eng_dev_categories = prepare_data(eng_dev)
eng_test_texts, eng_test_categories = prepare_data(eng_test)

# isiXhosa
xho_train_texts, xho_train_categories = prepare_data(xho_train)
xho_dev_texts, xho_dev_categories = prepare_data(xho_dev)
xho_test_texts, xho_test_categories = prepare_data(xho_test)

# chiShona
sna_train_texts, sna_train_categories = prepare_data(sna_train)
sna_dev_texts, sna_dev_categories = prepare_data(sna_dev)
sna_test_texts, sna_test_categories = prepare_data(sna_test)

In [22]:
def tokenise_text(text):
    return text.split()

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
eng_count_vectorizer = CountVectorizer(binary=False)

eng_train_bow = eng_count_vectorizer.fit_transform(eng_train_texts)
eng_dev_bow = eng_count_vectorizer.transform(eng_dev_texts)
eng_test_bow = eng_count_vectorizer.transform(eng_test_texts)

In [28]:
# Binary presence or absence
eng_binary_vectorizer = CountVectorizer(binary=True)

eng_train_binary = eng_binary_vectorizer.fit_transform(eng_train_texts)
eng_dev_binary = eng_binary_vectorizer.transform(eng_dev_texts)
eng_test_binary = eng_binary_vectorizer.transform(eng_test_texts)

# TF-IDF
eng_tfidf_vectorizer = TfidfVectorizer()

eng_train_tfidf = eng_tfidf_vectorizer.fit_transform(eng_train_texts)
eng_dev_tfidf = eng_tfidf_vectorizer.transform(eng_dev_texts)
eng_test_tfidf = eng_tfidf_vectorizer.transform(eng_test_texts)

In [29]:
# Binary presence or absence
eng_binary_vectorizer = CountVectorizer(binary=True)

eng_train_binary = eng_binary_vectorizer.fit_transform(eng_train_texts)
eng_dev_binary = eng_binary_vectorizer.transform(eng_dev_texts)
eng_test_binary = eng_binary_vectorizer.transform(eng_test_texts)

# TF-IDF
eng_tfidf_vectorizer = TfidfVectorizer()

eng_train_tfidf = eng_tfidf_vectorizer.fit_transform(eng_train_texts)
eng_dev_tfidf = eng_tfidf_vectorizer.transform(eng_dev_texts)
eng_test_tfidf = eng_tfidf_vectorizer.transform(eng_test_texts)

In [30]:
# Bag-of-Words counts
xho_count_vectorizer = CountVectorizer(binary=False)

xho_train_bow = xho_count_vectorizer.fit_transform(xho_train_texts)
xho_dev_bow = xho_count_vectorizer.transform(xho_dev_texts)
xho_test_bow = xho_count_vectorizer.transform(xho_test_texts)

# Binary presence or absence
xho_binary_vectorizer = CountVectorizer(binary=True)

xho_train_binary = xho_binary_vectorizer.fit_transform(xho_train_texts)
xho_dev_binary = xho_binary_vectorizer.transform(xho_dev_texts)
xho_test_binary = xho_binary_vectorizer.transform(xho_test_texts)

# TF-IDF
xho_tfidf_vectorizer = TfidfVectorizer()

xho_train_tfidf = xho_tfidf_vectorizer.fit_transform(xho_train_texts)
xho_dev_tfidf = xho_tfidf_vectorizer.transform(xho_dev_texts)
xho_test_tfidf = xho_tfidf_vectorizer.transform(xho_test_texts)

In [31]:
# Bag-of-Words counts
sna_count_vectorizer = CountVectorizer(binary=False)

sna_train_bow = sna_count_vectorizer.fit_transform(sna_train_texts)
sna_dev_bow = sna_count_vectorizer.transform(sna_dev_texts)
sna_test_bow = sna_count_vectorizer.transform(sna_test_texts)

# Binary presence or absence
sna_binary_vectorizer = CountVectorizer(binary=True)

sna_train_binary = sna_binary_vectorizer.fit_transform(sna_train_texts)
sna_dev_binary = sna_binary_vectorizer.transform(sna_dev_texts)
sna_test_binary = sna_binary_vectorizer.transform(sna_test_texts)

# TF-IDF
sna_tfidf_vectorizer = TfidfVectorizer()

sna_train_tfidf = sna_tfidf_vectorizer.fit_transform(sna_train_texts)
sna_dev_tfidf = sna_tfidf_vectorizer.transform(sna_dev_texts)
sna_test_tfidf = sna_tfidf_vectorizer.transform(sna_test_texts)

In [33]:
print("English")
print("Bag-of-Words:", eng_train_bow.shape)
print("Binary:", eng_train_binary.shape)
print("TF-IDF:", eng_train_tfidf.shape)

print("\nisiXhosa")
print("Bag-of-Words:", xho_train_bow.shape)
print("Binary:", xho_train_binary.shape)
print("TF-IDF:", xho_train_tfidf.shape)

print("\nchiShona")
print("Bag-of-Words:", sna_train_bow.shape)
print("Binary:", sna_train_binary.shape)
print("TF-IDF:", sna_train_tfidf.shape)

English
Bag-of-Words: (3309, 50456)
Binary: (3309, 50456)
TF-IDF: (3309, 50456)

isiXhosa
Bag-of-Words: (1032, 63116)
Binary: (1032, 63116)
TF-IDF: (1032, 63116)

chiShona
Bag-of-Words: (1288, 45173)
Binary: (1288, 45173)
TF-IDF: (1288, 45173)
